# 第52章 热力图（heatmap）

用颜色矩阵展示相关系数、透视表或任意二维数值。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

比较行×列组合或压缩读取数值矩阵。

## 数据结构

二维矩阵或可透视成长×宽矩阵的长表。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 cmap="vlag" 改为 cmap="coolwarm" 或 "RdBu_r"，对比不同发散色盘的视觉效果
2. 修改 center=0 为不设置 center，观察色阶中心对相关矩阵显示的影响
3. 调整 fmt=".2f" 为 fmt=".0f"，说明标注精度对数值可读性的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(36)
n = 240
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], n, p=[0.34, 0.38, 0.28]),
    "channel": rng.choice(["自然流量", "广告", "会员"], n, p=[0.42, 0.36, 0.22]),
    "region": rng.choice(["华东", "华南", "华北"], n),
    "order_value": np.clip(rng.normal(260, 72, n), 45, None),
    "items": rng.integers(1, 7, n),
})
orders.loc[orders["category"] == "数码", "order_value"] *= 1.35
orders["satisfied"] = rng.choice(["满意", "一般"], n, p=[0.78, 0.22])

marketing = pd.DataFrame({
    "channel": rng.choice(["搜索", "社交", "会员"], n),
    "visits": rng.integers(80, 850, n),
    "ad_spend": rng.uniform(2, 38, n),
})
marketing["sales"] = (
    45 + marketing["visits"] * 0.16 + marketing["ad_spend"] * 2.4
    + marketing["channel"].map({"搜索": 18, "社交": 8, "会员": 32})
    + rng.normal(0, 28, n)
).clip(10)
marketing["conversion"] = (marketing["sales"] / marketing["visits"]).clip(0.02, 0.5)

daily = pd.DataFrame({
    "date": np.tile(pd.date_range("2026-01-01", periods=12, freq="D"), 3),
    "region": np.repeat(["华东", "华南", "华北"], 12),
})
daily["sales"] = (
    np.tile(np.linspace(110, 190, 12), 3)
    + np.repeat([28, 8, 18], 12)
    + rng.normal(0, 9, 36)
)

sns.set_theme(style="whitegrid", context="notebook")
print("订单样本:", orders.shape, "营销样本:", marketing.shape)


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
corr = marketing[["visits", "ad_spend", "sales", "conversion"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("营销指标相关系数")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
pivot = orders.pivot_table(index="region", columns="category", values="order_value", aggfunc="mean")
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues", linewidths=0.5, cbar_kws={"label": "平均客单价（元）"}, ax=ax)
ax.set(title="区域与品类客单价", xlabel="品类", ylabel="区域")
fig.tight_layout()
plt.show()


## 3. 参数说明

- annot：标注
- fmt：格式
- cmap：色盘
- center/vmin/vmax：色阶


## 4. 结果解读

先读色阶含义，再找极值、带状结构和异常组合。


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第52章 热力图（heatmap）”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## Seaborn 的学习主线

提出统计问题 → 整理成长表 → 明确 x、y、hue 和分组 → 绘制统计图 → 比较类别和分布 → 处理排序与不确定性 → 用文字解释结论

先确认每一行是一条观察，再决定图形统计什么。看到均值、箱体或置信区间时，要说明它们代表什么，不能只描述颜色和形状。


## 本模块练习方式

基础：完成一张分组图；提高：改变分组或排序并比较结论；挑战：同时保留摘要和原始点，说明样本量对解读的影响。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 示例 4：用长表完成一次分组绘图

这一组例子只处理一个小问题。先运行代码，再逐行对照拆解说明。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南", "华北", "华北"],
    "channel": ["线上", "线下", "线上", "线下", "线上", "线下"],
    "sales": [120, 90, 150, 110, 100, 80],
})
sns.set(style="whitegrid")
sns.barplot(data=df, x="region", y="sales", hue="channel", ci=None)
plt.title("地区与渠道销售额")
plt.show()


### 逐步拆解

长表的每一行是一条观察；x、y 和 hue 分别决定位置、数值和分类颜色。

建议第一次运行后只改一个输入值，再观察哪一个输出发生变化。


## 示例 5：保留原始观察再看摘要

看懂上一个例子后，再观察同一主题在另一种数据或场景中的写法。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南", "华北", "华北"],
    "sales": [120, 150, 100, 180, 90, 130],
})
sns.boxplot(data=df, x="region", y="sales", color="lightgray")
sns.stripplot(data=df, x="region", y="sales", color="#2563eb", alpha=0.7)
plt.title("摘要分布与原始点")
plt.show()


### 逐步拆解

箱线图提供摘要，散点保留真实记录；两者叠加适合样本量不太大的数据。

自我检查：如果把输入数量、类别或参数改成另一组值，代码是否仍然能运行？


## 教学实验：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({
    "region": ["华东", "华南", "华北", "西南"],
    "sales": [320, 250, 280, 190],
})
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E")
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B")
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 常见误区

- 色阶范围随数据变化无法跨图比较
- 相关矩阵使用单向色盘
- 标注过密


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
counts = pd.crosstab(orders["region"], orders["channel"])
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(counts, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5, ax=ax)
ax.set(title="区域与渠道订单量", xlabel="渠道", ylabel="区域")
fig.tight_layout()
plt.show()


## 本章小结

用颜色矩阵展示相关系数、透视表或任意二维数值。


### 你已经掌握

- 判断热力图（heatmap）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较行×列组合或压缩读取数值矩阵。 |
| 数据结构 | 二维矩阵或可透视成长×宽矩阵的长表。 |
| 结果解读 | 先读色阶含义，再找极值、带状结构和异常组合。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `annot` | 标注 |
| `fmt` | 格式 |
| `cmap` | 色盘 |
| `center/vmin/vmax` | 色阶 |


### 需要注意

- 色阶范围随数据变化无法跨图比较
- 相关矩阵使用单向色盘
- 标注过密


### 完成检查

- [ ] 能判断什么问题适合使用热力图（heatmap）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
